In [ ]:
#KOMÓRKA 1 i 2: PRZYGOTOWANIE DANYCH Z PRZEWOŹNIKIEM
!pip install -q pyspark kagglehub gradio

from pyspark.sql import SparkSession
import kagglehub
from pyspark.sql.functions import col, when, month, hour
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

spark = SparkSession.builder.appName("FlightDelay_Carrier").getOrCreate()

# Pobranie danych
path = kagglehub.dataset_download("williamparker20/flight-ontime-reporting-with-weather")
df = spark.read.option("recursiveFileLookup", "true").csv(path, header=True, inferSchema=True)

# 1. Wybór kolumn - DODAJEMY 'Carrier' (Przewoźnik)
cols_needed = ["DepDelayMinutes", "Time", "Origin", "Carrier", # <--- NOWOŚĆ
               "Temperature", "Wind_Speed", "Precipitation",
               "Wind_Gust", "Ice_Accretion_3hr"] # Te z poprzedniego kroku też zostawiamy

# Wypełniamy braki w pogodzie zerami, resztę usuwamy
df_clean = df.select(cols_needed).na.fill(0.0, ["Wind_Gust", "Ice_Accretion_3hr"]).dropna()

# 2. Tworzenie Etykiety
df_prepared = df_clean.withColumn("Label", when(col("DepDelayMinutes") > 15, 1.0).otherwise(0.0)) \
                      .withColumn("Month", month(col("Time"))) \
                      .withColumn("Hour", hour(col("Time")))

# BALANSOWANIE KLAS
delayed_count = df_prepared.filter(col("Label") == 1.0).count()
on_time_count = df_prepared.filter(col("Label") == 0.0).count()
balance_ratio = on_time_count / delayed_count
df_weighted = df_prepared.withColumn("classWeight", when(col("Label") == 1.0, balance_ratio).otherwise(1.0))

# 3. Indeksowanie
indexer_origin = StringIndexer(inputCol="Origin", outputCol="OriginIndex")
indexer_carrier = StringIndexer(inputCol="Carrier", outputCol="CarrierIndex") # <--- NOWOŚĆ

# 4. Składanie wektora cech
assembler = VectorAssembler(
    inputCols=["OriginIndex", "CarrierIndex", "Month", "Hour",
               "Temperature", "Wind_Speed", "Wind_Gust", "Precipitation", "Ice_Accretion_3hr"],
    outputCol="features"
)

train_data, test_data = df_weighted.randomSplit([0.8, 0.2], seed=42)
print("Dane przygotowane. Uwzględniono przewoźników i historię opóźnień.")

100%|██████████| 157M/157M [00:01<00:00, 133MB/s]

Extracting files...


Dane przygotowane. Uwzględniono przewoźników i historię opóźnień.


In [ ]:
#KOMÓRKA 3: TRENING
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Model Random Forest 
rf = RandomForestClassifier(labelCol="Label",
                            featuresCol="features",
                            weightCol="classWeight",
                            numTrees=50,
                            maxDepth=10,
                            maxBins=64)

pipeline = Pipeline(stages=[indexer_origin, indexer_carrier, assembler, rf])

print("Rozpoczynam trenowanie modelu z uwzględnieniem przewoźników...")
model = pipeline.fit(train_data)
print("Model gotowy!")

# Ewaluacja
predictions = model.transform(test_data)
evaluator = BinaryClassificationEvaluator(labelCol="Label")
auc = evaluator.evaluate(predictions)

print(f"\n>>> WYNIK AUC: {auc:.4f} <<<")

# Sprawdzenie ważności cech - czy Przewoźnik (Carrier) jest ważny?
importances = model.stages[-1].featureImportances
print("\nWażność cech (indeksy 0=Origin, 1=Carrier...):")
print(importances)

Rozpoczynam trenowanie modelu z uwzględnieniem przewoźników...
Model gotowy!

>>> WYNIK AUC: 0.6935 <<<

Ważność cech (indeksy 0=Origin, 1=Carrier...):
(9,[0,1,2,3,4,5,6,7,8],[0.043747016664573,0.1838666564334492,0.04447956330324883,0.4405495902128908,0.0762478902952985,0.011914387874154395,0.019837036967424705,0.17932146061944268,3.639762951789021e-05])


In [ ]:
#KOMÓRKA 4: FRONTEND
import gradio as gr
from pyspark.sql import Row

# Wyciągamy listy do dropdownów
origins = model.stages[0].labels
carriers = model.stages[1].labels # Lista linii lotniczych
origins.sort()
carriers.sort()

def predict_full(carrier, airport, month, hour, temp, wind, gust, precip, ice):
    try:
        user_data = spark.createDataFrame([
            Row(
                Carrier=carrier,         
                Origin=airport,
                Month=int(month),
                Hour=int(hour),
                Temperature=float(temp),
                Wind_Speed=float(wind),
                Wind_Gust=float(gust),
                Precipitation=float(precip),
                Ice_Accretion_3hr=float(ice)
            )
        ])

        pred = model.transform(user_data)
        prob = pred.select("probability").collect()[0][0][1]

        # Interpretacja wyniku
        risk_level = "WYSOKIE" if prob > 0.6 else "ŚREDNIE" if prob > 0.4 else "NISKIE"
        return f"Ryzyko opóźnienia: {risk_level} ({prob:.1%})\nAnaliza dla linii {carrier} z lotniska {airport}."

    except Exception as e:
        return f"Błąd: {str(e)}"

iface = gr.Interface(
    fn=predict_full,
    inputs=[
        gr.Dropdown(choices=carriers, label="Linia Lotnicza (Carrier)", value=carriers[0]), 
        gr.Dropdown(choices=origins, label="Lotnisko Wylotu", value=origins[0]),
        gr.Slider(1, 12, label="Miesiąc", value=1),
        gr.Slider(0, 23, label="Godzina", value=12),
        gr.Slider(-10, 100, label="Temperatura (F)"),
        gr.Slider(0, 40, label="Wiatr (mph)"),
        gr.Slider(0, 60, label="Porywy wiatru (Gust)"),
        gr.Slider(0, 2, label="Opady"),
        gr.Slider(0, 5, label="Oblodzenie")
    ],
    outputs="text",
    title="Kalkulator Ryzyka Opóźnienia Lotu",
    description="Model oparty na historycznych danych przewoźników i warunkach pogodowych."
)

iface.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1060a63ac7ed806019.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
